# Case Study: Bayesian logistic regression with a Laplace prior via SOUL with Metropolis-Hastings step.

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))  
sys.path.append(project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from algorithms import sig, log_p_laplace, so_mh, so_mh_decay
#pip_ula, prox_pgd, proximal_map_laplace_iteration_total, pipgla, proximal_map_laplace_approx_total
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

os.chdir(project_root)

# Obtain synthetic dataset for Laplace prior case

In [ ]:
from scipy.stats import laplace, bernoulli
# Data and design matrix
np.random.seed(3)
design_matrix = np.random.uniform(low = -1.0, high = 1.0, size = (900, 50))
x_unknown = laplace.rvs(loc = -4, size = (50, 1))
parameter_bernoulli = sig(np.matmul(design_matrix, x_unknown))
data_experiment = bernoulli.rvs(np.array(parameter_bernoulli[:, 0]), size = 900)
labels = np.expand_dims(data_experiment, axis=1)
theta_true = -4

# SOUL Metropolis-Hastings algorithm 

In [ ]:
#@title One run test
# --- SOUL Optimization Setup ---
D = 50       # Dimensionality of latent variables (x)
T = 800      # Outer optimization steps
M = 35       # MH steps per outer loop
B = 5       # Burn-in steps
delta_step = 0.08
proposal_std = 0.05
b_scale = 1.0 # Scale parameter for the Laplace prior

# Initializations
th0 = np.array([[-15.0]]) 
x0_M = np.zeros((D, 1))   # Initial latent variables vector

# Run the adapted algorithm
th_list, x_samples = so_mh(
    log_p=log_p_laplace,
    th0=th0,
    x0_M=x0_M,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    D=D,
    delta_step=delta_step,
    proposal_std=proposal_std,
    b=b_scale
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

In [ ]:
# This is not parallelized -- use  for debugging and comment out

"""
# --- Experiment Parameters ---
T = 500  # Outer steps 
B = 5               
M = 20 

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.08    # Theta learning rate
proposal_std = 0.05  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale

fig = plt.figure(figsize=(10, 6))
theta_soul = []
X_soul = []

for run_idx in range(7):
    # Randomly initialize theta0 between -15 and 10
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    
    # Initialize X0 drawn from Gaussian centered at theta0, shape (D, M)
    # where D is feature dimension (50 from Paula's dataset)
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))
    
    th_list, x_values = so_mh(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale
    )
    
    # Extract flattened array of theta trajectory across iterations
    th_trajectory = np.array([t[0, 0] for t in th_list])
    
    theta_soul.append(th_trajectory)
    X_soul.append(x_values)
    
    # Plot trajectory for this run
    plt.plot(th_trajectory, label=f'Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)')

# Reference line for ground truth mean
plt.axhline(y=np.mean(x_unknown), color='black', linestyle='dashed', label='True Mean $\\theta$')

plt.title('SOUL-MH Convergence Across Multiple Initializations')
plt.xlabel('Iteration (T)')
plt.ylabel('Theta Estimate ($\\theta$)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()
"""

Because all runs are independent, let's parallelise them to optimise speed:

In [ ]:
pip install joblib

In [ ]:
from joblib import Parallel, delayed

# tried parallelisation to go faster ! success :))

# --- Experiment Parameters ---
T = 2500  # Outer steps 
B = 5               
M = 40

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.05    # Theta learning rate
proposal_std = 0.08  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale
gamma_val = 0.9995

thetas = [0.0, 3.0, -13.0, -6.0, 8.0, -4.0, 5.0]

def run_single_soul_mh_decay(run_idx, thetas):
    theta0_val = thetas[run_idx]
    th0 = np.array([[float(theta0_val)]])
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))

    th_list, x_values = so_mh_decay(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale,
        gamma = gamma_val
    )
    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values


# Run all 7 initializations in parallel across available CPU cores (leave at least one free so the computer stays responsive)
results = Parallel(n_jobs=-2)(
    delayed(run_single_soul_mh_decay)(i, thetas) for i in range(len(thetas))
)

# Plot all 7 runs on one figure
fig = plt.figure(figsize=(10, 6))

theta_mh = []
X_mh = []

# Loop over the parallel results and add each line to the same plot
for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_mh.append(th_trajectory)
    X_mh.append(x_values)

    plt.plot(
        th_trajectory,
        label=f"Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)",
    )

# Reference line for ground truth
plt.axhline(
    y=np.mean(x_unknown),
    color="black",
    linestyle="dashed",
    label="True Mean $\\theta$",
)

plt.title("SOMH Convergence Across Multiple Initializations ") 
plt.xlabel("Iteration (T)")
plt.ylabel("Theta Estimate ($\\theta$)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.show()


The relative error:

In [ ]:
# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_mh_arr = np.array(theta_mh)

# Compute relative error trajectory for each run
relative_error_trajectories = np.abs(theta_mh_arr - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step = np.mean(relative_error_trajectories, axis=0)

# Plot Relative Error over iterations
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOUL-MH Relative Error Convergence")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

- Steady Initial Descent: The mean relative error (black line) decreases consistently across the first ~800 iterations, dropping from above $1.0$ down to around $0.2$ ($20\%$ error)
- Error Plateau: Around $T = 800$, the mean error hits a plateau at roughly $0.2$ ($20\%$). It stops improving systematically and begins oscillating.


# Error with the same $\theta_0$

In [ ]:
from joblib import Parallel, delayed

# --- Experiment Parameters --- same by default
T = 1000

"""
B = 5               
M = 40

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.05    # Theta learning rate
proposal_std = 0.08  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale
gamma_val = 0.9995
"""
thetatest = 3.0

def run_single_somh_fix_decay(run_idx, theta0_val):
    th0 = np.array([[float(theta0_val)]])
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))

    th_list, x_values = so_mh_decay(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale,
        gamma = gamma_val
    )
    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values

# Run all 5 initializations in parallel across available CPU cores (leave at least one free so the computer stays responsive)
results = Parallel(n_jobs=-2)(
    delayed(run_single_somh_fix_decay)(i, thetatest) for i in range(5)
)

# Plot all 5 runs on one figure
fig = plt.figure(figsize=(10, 6))

theta_mh_same = []
X_mh_same = []

# Loop over the parallel results 
for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_mh_same.append(th_trajectory)
    X_mh_same.append(x_values)

# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_mh_arr_same = np.array(theta_mh_same)

# Compute relative error trajectory for each run
relative_error_trajectories_same = np.abs(theta_mh_arr_same - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step_same = np.mean(relative_error_trajectories_same, axis=0)

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories_same):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step_same,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOMH Relative Error Convergence ($theta_0$ = 3.0)")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# Save data 

In [ ]:
# Save trajectories, ground truth, and precalculated errors
np.savez_compressed(
    "so_mh_results.npz",
    T=T,
    B=B,
    M=M,
    delta_step=delta_step,
    gamma_val = gamma_val,
    proposal_std= proposal_std,
    b_scale = b_scale,
    theta_true=theta_true,
    theta_mh=theta_mh_arr,
    X_mh=np.array(X_mh),
    relative_error_trajectories=relative_error_trajectories,
    mean_relative_error_per_step=mean_relative_error_per_step,
    X_mh_same=np.array(X_mh_same),
    theta_mh_same=theta_mh_arr_same,
    relative_error_trajectories_same=relative_error_trajectories_same,
    mean_relative_error_per_step_same=mean_relative_error_per_step_same
)

print("Data saved successfully to 'so_mh_results.npz'")

# Load compressed data

In [ ]:
# Load compressed data
data = np.load("so_mh_results.npz")

T = data["T"]
theta_true = data["theta_true"]
theta_mh_arr = data["theta_mh"]
relative_error_trajectories = data["relative_error_trajectories"]
mean_relative_error_per_step = data["mean_relative_error_per_step"]

# Plot 1: Theta Convergence Trajectories ---
plt.figure(figsize=(10, 6))

for idx, th_trajectory in enumerate(theta_mh_arr):
    plt.plot(th_trajectory, alpha=0.7, label=f"Run {idx + 1}")

plt.axhline(
    y=theta_true,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=r"True $\theta$",
)

plt.title("SOMH Convergence Across Multiple Initializations", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel(r"Theta Estimate ($\theta$)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Plot 2: Relative Error Convergence (Log Scale) ---
plt.figure(figsize=(10, 5))

for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.3, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)

plt.yscale("log")
plt.title("SOMH Relative Error Convergence", fontsize=14, pad=12)
plt.xlabel("Iteration ($T$)", fontsize=12)
plt.ylabel("Relative Error (Log Scale)", fontsize=12)
plt.legend(loc="upper right", frameon=True, fontsize=10)
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()